# Import

In [1]:
from scipy.io import loadmat
import pandas as pd
import os
import matplotlib.pyplot as plt
import numpy as np
from mapd import Trial, Table, Sinq
import h5py
%load_ext autoreload
%autoreload 2

%matplotlib widget

def refresh_table_addons():
    import importlib
    import mapd.table_plotters as tps
    import mapd.table_scalars as tbs
    importlib.reload(tps)
    importlib.reload(tbs)

    for name in dir(tps):
        if name.startswith('_'):
            continue
        attr = getattr(tps, name)
        setattr(Table, name[len("plot_"):], attr)
    
    for name in dir(tbs):
        if name.startswith('_'):
            continue
        if name.startswith('compute_'):
            attr = getattr(tbs, name)
            setattr(Table, name[len("compute_"):], attr)

RECOMPUTE = True

# Create Figure 4 sinq - MN spiking

In [2]:
sinq = Sinq(sinqname='Figure4')
print(sinq.__repr__())

sinq.display_df()

Found data directory: D:\Data
Sinq(Figure4, 13 rows x 30 params): 0 empties; ['210302_F1_C2', '210319_F2_C1', '210331_F2_C1', '210405_F1_C1', '210602_F1_C1', '210604_F1_C1', '210903_F3_C1', '210908_F3_C1', '210915_F1_C1', '210917_F2_C1', '240430_F1_C1', '241115_F1_C1', '241203_F2_C1'] x ['Table', 'parquet', 'genotype', 'notes', 'duration', 'rms_velocity', 'outcome_fractions_no_as_no_mv', 'outcome_fractions_as_off', 'outcome_fractions_rest', 'outcome_fractions_no_as_mv', 'outcome_fractions_probe', 'outcome_fractions_as_off_late', 'outcome_fractions_timeout_fail', 'outcome_fractions_timeout', 'outcome_fractions_info', 'lo_state_median_position', 'hi_state_median_position', 'hi_lo_shift', 'hi_state_on_target', 'lo_state_on_target', 'num_trials', 'blue_fraction', 'blue_toggle_fraction', 'most_common_fiberLED', 'lo_target_off_state', 'hi_target_off_state', 'probe_positive_effort', 'successes', 'hard_successes', 'holding_cost'] 


,Table,parquet,genotype,notes,note_text,tags,duration,rms_velocity,outcome_fractions_no_as_no_mv,outcome_fractions_as_off,...,num_trials,blue_fraction,blue_toggle_fraction,most_common_fiberLED,lo_target_off_state,hi_target_off_state,probe_positive_effort,successes,hard_successes,holding_cost
210302_F1_C2,None,LEDFlashWithPiezoCueControl_210302_F1_C2_Table...,Hot-Cell-Gal4 (test),"{'text': '', 'tags': ['control', 'nonmotor', '...",,"control, nonmotor, random",6836.72718,32982.968041,0.561250,0.173750,...,800.0,1.0,0.000000,epi_only,0.055824,0.082934,8.684294e+05,78.0,37.0,9.147350e+06
210319_F2_C1,None,LEDFlashWithPiezoCueControl_210319_F2_C1_Table...,Hot-Cell-LexA_Chr;81A06_pJFRC7,"{'text': '', 'tags': ['F1', 'periodic_input'],...",,"F1, periodic_input",6141.59006,63494.653833,0.236697,0.231193,...,545.0,1.0,0.239583,epi_only,0.049627,0.018823,1.655564e+06,58.0,18.0,5.410378e+06
210331_F2_C1,None,LEDFlashWithPiezoCueControl_210331_F2_C1_Table...,+;Hot-Cell-LexA_Chr;81A06-GAL4_pJFRC7,"{'text': '', 'tags': ['F1', 'period input'], '...",,"F1, period input",8326.59290,123912.688076,0.341689,0.188654,...,758.0,1.0,0.218563,625_red,0.353006,0.094575,3.277774e+06,38.0,13.0,8.579387e+06
210405_F1_C1,None,LEDFlashWithPiezoCueControl_210405_F1_C1_Table...,+;Hot-Cell-LexA_Chr;81A06-GAL4_pJFRC7,"{'text': '', 'tags': ['C1'], 'author': 'Tony',...",,C1,6582.81488,34006.653998,0.462382,0.087774,...,638.0,1.0,0.171378,625_red,0.145846,0.206347,1.349486e+06,34.0,13.0,1.307052e+07
210602_F1_C1,None,LEDFlashWithPiezoCueControl_210602_F1_C1_Table...,+;Hot-Cell-LexA_Chr;35C09-GAL4_pJFRC7,"{'text': '', 'tags': ['A2', 'slow'], 'author':...",,"A2, slow",5282.20700,76792.369491,0.415385,0.035385,...,650.0,1.0,0.254417,625_red,0.217067,0.376011,1.130179e+06,26.0,5.0,7.129659e+06
210604_F1_C1,None,LEDFlashWithPiezoCueControl_210604_F1_C1_Table...,+;Hot-Cell-LexA_Chr;35C09-GAL4_pJFRC7,"{'text': '', 'tags': ['A2', 'slow'], 'author':...",,"A2, slow",3685.79266,44418.935879,0.565714,0.102857,...,350.0,1.0,0.107143,625_red,0.138731,0.180145,1.728146e+06,15.0,9.0,6.519351e+06
210903_F3_C1,None,LEDFlashTriggerPiezoControl_210903_F3_C1_Table...,+;Hot-Cell-LexA_Chr;35C09-GAL4_pJFRC7,"{'text': '', 'tags': ['A2', 'learn', 'slow'], ...",,"A2, learn, slow",6303.51030,45596.022785,0.426667,0.131667,...,600.0,1.0,0.204545,625_red,0.125386,0.072073,1.898442e+06,37.0,14.0,1.363618e+07
210908_F3_C1,None,LEDFlashTriggerPiezoControl_210908_F3_C1_Table...,+;Hot-Cell-LexA_Chr;35C09-GAL4_pJFRC7,"{'text': '', 'tags': ['A2', 'slow'], 'author':...",,"A2, slow",2210.02624,26621.868106,0.483412,0.146919,...,211.0,1.0,0.182320,625_red,0.318175,0.073519,1.106503e+06,13.0,5.0,6.160116e+06
210915_F1_C1,None,LEDFlashTriggerPiezoControl_210915_F1_C1_Table...,+;Hot-Cell-LexA_Chr;78E05-GAL4_pJFRC7,"{'text': '', 'tags': ['A2', 'bright_cell'], 'a...",,"A2, bright_cell",5857.48988,45659.433547,0.410463,0.084507,...,497.0,1.0,0.237986,625_red,0.109020,0.279235,2.327835e+06,27.0,12.0,1.325873e+07
210917_F2_C1,None,LEDFlashTriggerPiezoControl_210917_F2_C1_Table...,+;Hot-Cell-LexA_Chr;31H05-GAL4_pJFRC7,"{'text': '', 'tags': ['A4', 'dim_cell'], 'auth...",,"A4, dim_cell",5780.00094,74683.629443,0.315498,0.276753,...,542.0,1.0,0.182773,625_red,0.111971,0.241052,4.262578e+06,50.0,19.0,1.208222e+07


# Whole cell recordings

### 210302_F1_C2

In [ ]:
# id = '210302_F1_C2'
# if not sinq.df is None and id in sinq.df.index:
#     T = sinq.restore_table(dayflycell=id,overwrite=True)
#     # endofdaytrials = T.df.loc[1081:]
#     # T.exclude_list_of_trials(endofdaytrials.index,reason='tired, frustrated poor behavior')
# else:
#     T = Table(f'LEDFlashWithPiezoCueControl_{id}_Table.parquet')
#     T.assign_column_value('filtercube_status','blue',trial_min=1, write_to_hdf5=True)
#     T.assign_column_value('fiberLED','epi_only',trial_min =1,write_to_hdf5=True)
#     sinq.add_table(table=T)
#     sinq.df
# T.exclude_trials()
# T.extract_trial_properties()
# T.find_successful_trials()
# fig,ax = T.plot_outcomes(savefig=True,format='png')
# fig,ax,df = T.plot_probe_position_heatmap(cmin=-500,cmax=10,format='png')

In [ ]:
id = '210302_F1_C2'
T = Table(f'LEDFlashWithPiezoCueControl_{id}_Table.parquet')
sinq.clear_note(dayflycell=id)
sinq.set_note(
        dayflycell = id,
        text= 'premn',
        tags=['nonmotor','random','control'],
        append=False,
)


In [ ]:
sinq.display_df()

### 210319_F2_C1 - Do not use
What is the deal!? There is an enormous number of points and the distributions is near probe_Zero, but the heatmap shows displacement. Not clear where this is coming from...?

In [ ]:
# T = Table('LEDFlashWithPiezoCueControl_210319_F2_C1_Table.parquet',progress_bar=True,add_probestate=False)
# # Found 4 target positions - Counter({('lo', 730.0, -180.0, 60.0): 251, ('hi', 730.0, -260.0, 60.0): 199, ('no_state', 0.0, 520.0, 80.0): 50, ('no_state', 0.0, 450.0, 80.0): 50})
# # T.df.loc[T.df['pyasXPosition']==490,'Trial'].index
# # T.exclude_list_of_trials(T.df.loc[T.df['pyasXPosition']==490,'Trial'].index,reason='unused target position')
# # T.df.loc[T.df['pyasXPosition']==520,'probeZero'] = 730
# # T.df.loc[T.df['pyasXPosition']==450,'probeZero'] = 730
# # T.df.loc[T.df['pyasXPosition']==520,'pyasState'] = 'lo'
# # T.df.loc[T.df['pyasXPosition']==450,'pyasState'] = 'hi'
# # T.df.loc[T.df['pyasXPosition']==450,'pyasState']
# # T.write_column_to_trial_files('pyasState')
# # T.write_column_to_trial_files('probeZero')
# T.assign_column_value('filtercube_status','blue',trial_min =1, write_to_hdf5=True)
# T.assign_column_value('fiberLED','625_red',trial_min =1,write_to_hdf5=True)
# T.extract_trial_properties()
# fig,ax = T.plot_outcomes(savefig=True,format='png')
# fig,ax,df = T.plot_probe_position_heatmap(cmin=-500,cmax=10,format='png')
# trial = Trial('LEDFlashWithPiezoCueControl_Raw_210319_F2_C1_1.mat')
# trial.fiberLED

In [ ]:
# id = '210319_F2_C1'
# if not sinq.df is None and id in sinq.df.index:
#     T = sinq.restore_table(dayflycell=id,overwrite=True)
#     # endofdaytrials = T.df.loc[1081:]
#     # T.exclude_list_of_trials(endofdaytrials.index,reason='tired, frustrated poor behavior')
# else:
#     T = Table(f'LEDFlashWithPiezoCueControl_{id}_Table.parquet')
#     T.assign_column_value('filtercube_status','blue',trial_min=1, write_to_hdf5=True)
#     T.assign_column_value('fiberLED','epi_only',trial_min =1,write_to_hdf5=True)
#     sinq.add_table(table=T)
#     sinq.df
# T.exclude_trials()
# T.extract_trial_properties()
# T.find_successful_trials()
# fig,ax = T.plot_outcomes(savefig=True,format='png')
# fig,ax,df = T.plot_probe_position_heatmap(cmin=-500,cmax=10,format='png')

In [ ]:
id = '210319_F2_C1'
# T = Table(f'LEDFlashWithPiezoCueControl_{id}_Table.parquet')
# T.df.loc[T.df['probeZero']==730,'Trial'].index
# T.df.loc[T.df['probeZero']==730,'probeZero'] = 690 
# T.write_column_to_trial_files('probeZero')

sinq.clear_note(dayflycell=id)
sinq.set_note(
        dayflycell = id,
        text= '81A06',
        tags=['F1','periodic_input'],
        append=False,
)


In [ ]:
sinq.drop_tables()

### 210331_F2_C1 - HC-LexA/13XLexAop-ChrimsonR;81A06/pJFRC7 100uL ATR mixed in

In [ ]:
id = '210331_F2_C1'
# T = Table(f'LEDFlashWithPiezoCueControl_{id}_Table.parquet')
# sinq.add_table(table=T)
# T.open_notes_files()
# endofdaytrials = T.df.loc[765:774]
# T.exclude_list_of_trials(endofdaytrials.index,reason='bad recordings')

# random_bad_trials = T.df.loc[[293, 295, 455, 621,625,626,700]]
# T.exclude_list_of_trials(random_bad_trials.index,reason='random bad probe recordings')

# sinq.clear_note(dayflycell=id)
# sinq.set_note(
#         dayflycell = id,
#         text= '81A06',
#         tags=['F1', 'period input'],
#         append=False,
# )
# sinq.display_df()

In [ ]:
id = '210405_F1_C1'
# T = Table(f'LEDFlashWithPiezoCueControl_{id}_Table.parquet')
# sinq.add_table(table=T)
# T.open_notes_files()


# sinq.set_note(
#         dayflycell = id,
#         text= '81A06',
#         tags=['C1'],
#         append=False,
# )

In [ ]:
id = '210602_F1_C1'
T = Table(f'LEDFlashWithPiezoCueControl_{id}_Table.parquet')
# Found 7 target positions - Counter({('lo', 550.0, 490.0, -60.0, 60.0): 241, 
#                                     ('hi', 550.0, 390.0, -160.0, 60.0): 200, 
#                                     ('no_state', 550.0, 310.0, -240.0, 60.0): 100, 
#                                     ('no_state', 550.0, 410.0, -140.0, 60.0): 51, 
#                                     ('no_state', 550.0, 380.0, -170.0, 60.0): 50, 
#                                     ('no_state', 550.0, 240.0, -310.0, 60.0): 7, 
#                                     ('no_state', 550.0, 140.0, -410.0, 60.0): 1})

# T.df.loc[T.df['pyasXPosition']==490.0,'Trial'].indexsinqsinssaswessidfwsighw
# T.df.loc[T.df['pyasXPosition']==380.0,'probeZero'].value_counts()
# # T.exclude_list_of_trials(T.df.loc[T.df['pyasXPosition']==490,'Trial'].index,reason='unused target position')
# T.df.loc[T.df['pyasXPosition']==490,'probeZero'] = 550  # not 670
# T.df.loc[T.df['pyasXPosition']==390,'probeZero'] = 550  # not 670
# T.write_column_to_trial_files('pyasState')
# T.write_column_to_trial_files('probeZero')
# T.df.loc[T.df['pyasXPosition']==490,'pyasState'] = 'lo'
# T.df.loc[T.df['pyasXPosition']==390,'pyasState'] = 'hi'


# T.df.loc[T.df['probeZero'].isna(),'Trial'].index
# T.df.loc[T.df['probeZero']==550,'probeZero'] = 570

# T.assign_column_value('filtercube_status','blue',trial_min =1, write_to_hdf5=True)
# T.assign_column_value('fiberLED','625_red',trial_min =1,write_to_hdf5=True)

# # sinq.add_table(table=T)
# T.open_notes_files()

sinq.set_note(
        dayflycell = id,
        text= '35C09_A2',
        tags=['A2','slow'],
        append=False,
)

In [ ]:
id = '210604_F1_C1'
# T = Table(f'LEDFlashWithPiezoCueControl_{id}_Table.parquet')
# sinq.add_table(table=T)
# T.open_notes_files()

# sinq.set_note(
#         dayflycell = id,
#         text= '35C09_A2',
#         tags=['A2','slow'],
#         append=False,
# )

# badtrials = T.df.loc[330:350]
# T.exclude_list_of_trials(badtrials.index,reason='Something is happening with the probe')


In [ ]:
id = '210903_F3_C1'
# T = Table(f'LEDFlashTriggerPiezoControl_{id}_Table.parquet')
# sinq.add_table(table=T)
# sinq.set_note(
#         dayflycell = id,
#         text= '35C09_A2',
#         tags=['learn','A2','slow'],
#         append=False,
# )
sinq.display_df()

In [ ]:
id = '210908_F3_C1'
# T = Table(f'LEDFlashTriggerPiezoControl_{id}_Table.parquet')
# Found 2 target positions - Counter({(495.0, 60.0): 106, (395.0, 60.0): 105})
# T.df.loc[T.df['pyasXPosition']==495.0,'Trial'].index
# T.df.loc[T.df['pyasXPosition']==495,'probeZero'] = 675
# T.df.loc[T.df['pyasXPosition']==395,'probeZero'] = 675
# T.df.loc[T.df['pyasXPosition']==495,'pyasState'] = 'lo'
# T.df.loc[T.df['pyasXPosition']==395,'pyasState'] = 'hi'
# T.df.loc[T.df['pyasXPosition']==395,'pyasState']
# T.write_column_to_trial_files('pyasState')
# T.write_column_to_trial_files('probeZero')

# T.assign_column_value('filtercube_status','blue',trial_min=1, write_to_hdf5=True)
# T.assign_column_value('fiberLED','625_red',trial_min =1,write_to_hdf5=True)

# endofdaytrials = T.df.loc[294:]
# T.exclude_list_of_trials(endofdaytrials.index,reason='bad recordings')

# sinq.add_table(table=T)
# sinq.set_note(
#         dayflycell = id,
#         text= '35C09_A2',
#         tags=['A2','slow'],
#         append=False,
# )
sinq.df
# T.open_notes_files()

In [ ]:
id = '210915_F1_C1'
# T = Table(f'LEDFlashTriggerPiezoControl_{id}_Table.parquet')
# sinq.add_table(table=T)
# T.open_notes_files()

# sinq.set_note(
#         dayflycell = id,
#         text= '78E05_bright_cell_A2',
#         tags=['bright_cell','A2'],
#         append=False,
# )

T = sinq.restore_table(dayflycell=id)

# use up to trial 
# endofdaytrials = T.df.loc[447:]
# T.exclude_list_of_trials(endofdaytrials.index,reason='first lost probe, then bad spikes')

# interesting_trials = {366:'Sudden extension'}



In [ ]:
id = '210917_F2_C1'
# T = Table(f'LEDFlashTriggerPiezoControl_{id}_Table.parquet')
# sinq.add_table(table=T)
sinq.set_note(
        dayflycell = id,
        text= '31H05_dim_cell_A4',
        tags=['dim_cell','A4'],
        append=False,
)

interesting_trials = {5:'high spike rate, big movements'}
sinq.display_df()

In [ ]:
id = '240430_F1_C1'
# T = Table(f'LEDFlashTriggerPiezoControl_{id}_Table.parquet')

# # Found 4 target positions - Counter({('lo', 730.0, -180.0, 60.0): 251, ('hi', 730.0, -260.0, 60.0): 199, ('no_state', 0.0, 520.0, 80.0): 50, ('no_state', 0.0, 450.0, 80.0): 50})
# Found 4 target positions - Counter({(540.0, 60.0): 347, (480.0, 60.0): 302, (440.0, 60.0): 38, (640.0, 60.0): 1})
# Need to write to pyasState
# T.df.loc[T.df['pyasXPosition']==440,'Trial'].index
# T.exclude_list_of_trials(T.df.loc[T.df['pyasXPosition']==640,'Trial'].index,reason='unused target position')
# T.df.loc[T.df['pyasXPosition']==540,'probeZero'] = 720
# T.df.loc[T.df['pyasXPosition']==480,'probeZero'] = 720
# T.df.loc[T.df['pyasXPosition']==440,'probeZero'] = 720
# T.df.loc[T.df['pyasXPosition']==540,'pyasState'] = 'lo'
# T.df.loc[T.df['pyasXPosition']==480,'pyasState'] = 'hi'
# # T.df.loc[T.df['pyasXPosition']==450,'pyasState']
# T.write_column_to_trial_files('pyasState')
# T.write_column_to_trial_files('probeZero')

# T.assign_column_value('filtercube_status','blue',trial_min=1, write_to_hdf5=True)
# T.assign_column_value('fiberLED','epi_only',trial_min =1,write_to_hdf5=True)

# use up to trial 290
# endofdaytrials = T.df.loc[290:]
# T.exclude_list_of_trials(endofdaytrials.index,reason='bad spikes')

# sinq.add_table(table=T)

sinq.set_note(
        dayflycell = id,
        text= '31H05_bright_cell_A2',
        tags=['dim_cell','A2'],
        append=False,
)

In [ ]:
id = '241115_F1_C1'
# T = Table(f'LEDFlashTriggerPiezoControl_{id}_Table.parquet')
# badtrials = T.df.loc[439:443]
# T.exclude_list_of_trials(badtrials.index,reason='poor data extraction, probably not captures on contacq')
# T.assign_column_value('filtercube_status','blue',trial_min=1, write_to_hdf5=True)
# T.assign_column_value('fiberLED','epi_only',trial_min =1,write_to_hdf5=True)
# sinq.add_table(table=T)

# sinq.set_note(
#         dayflycell = id,
#         text= '31H05_dim_cell_A4',
#         tags=['dim_cell','A4'],
#         append=False,
# )
# T = sinq.restore_table(dayflycell=id,overwrite=True)

In [5]:
id = '241203_F2_C1'
    # endofdaytrials = T.df.loc[1081:]
    # T.exclude_list_of_trials(endofdaytrials.index,reason='tired, frustrated poor behavior')
# else:
    # T = Table(f'LEDFlashTriggerPiezoControl_{id}_Table.parquet')
    # T.assign_column_value('filtercube_status','blue',trial_min=1, write_to_hdf5=True)
    # T.assign_column_value('fiberLED','epi_only',trial_min =1,write_to_hdf5=True)
    # sinq.add_table(table=T)
    # sinq.df

# T = sinq.restore_table(dayflycell=id,overwrite=True)

# T.exclude_trials()
# T.extract_trial_properties()
# T.find_successful_trials()

# sinq.set_note(
#         dayflycell = id,
#         text= '31H05_dim_cell_A4',
#         tags=['dim_cell','A4'],
#         append=False,
# )
# letgotrials = T.df.loc[201:204]
# T.exclude_list_of_trials(letgotrials.index,reason='let go of probe, had to put it back')
T = sinq.restore_table(dayflycell=id,overwrite=True)

Found data directory: D:\Data
T = pd.read_parquet("D:\\Data\\241203\\241203_F2_C1\\LEDFlashTriggerPiezoControl_241203_F2_C1_Table.parquet")
Getting trials
Excluding trials:
[Trial(trial=201, 241203_F2_C1, dT=7.08964, ex=True), Trial(trial=202, 241203_F2_C1, dT=7.33654, ex=True), Trial(trial=203, 241203_F2_C1, dT=7.2537, ex=True), Trial(trial=204, 241203_F2_C1, dT=7.10296, ex=True), Trial(trial=481, 241203_F2_C1, dT=10.33332, ex=True), Trial(trial=482, 241203_F2_C1, dT=10.93256, ex=True), Trial(trial=483, 241203_F2_C1, dT=9.86016, ex=True), Trial(trial=484, 241203_F2_C1, dT=7.68258, ex=True), Trial(trial=485, 241203_F2_C1, dT=10.00632, ex=True), Trial(trial=486, 241203_F2_C1, dT=7.57624, ex=True), Trial(trial=487, 241203_F2_C1, dT=7.66654, ex=True), Trial(trial=488, 241203_F2_C1, dT=9.67536, ex=True), Trial(trial=489, 241203_F2_C1, dT=8.4744, ex=True), Trial(trial=490, 241203_F2_C1, dT=9.73068, ex=True), Trial(trial=491, 241203_F2_C1, dT=10.28612, ex=True), Trial(trial=492, 241203_F2_C1

In [ ]:
trial = T.df.loc[24,'Trial']
trial.spikes_uncorrected

In [ ]:
sinq.df